In [5]:
from sklearn.model_selection import train_test_split
import pandas as pd

df_normalized = pd.read_csv('normalized_min-max.csv') 
target_normalized = df_normalized['AgreeSubsequentBooster']


features_normalized = df_normalized.drop('AgreeSubsequentBooster', axis=1)

x_train_normalized, x_test_normalized, y_train_normalized, y_test_normalized = train_test_split(features_normalized,
                                                                                                target_normalized,
                                                                                                test_size=0.2,
                                                                                                random_state=0)

In [6]:
anova_columns = [
    'Age','EmploymentYears',

    'S.VaccineImportantHealth','S.VaccineEffectiveSevere','S.VaccineImportantPublic',
    'S.VaccineBeneficialMOH','S.VaccineSafe','S.MOHInfoTrusted',
    'S.VaccineProtectInfection','S.AgreeMOHRecommendation','S.WorriedSideEffects',
    'S.PreventNewVariant','P.ProtectFlu','P.ProtectSevereComp','P.EffectivenessVsRisk',
    'P.ClinicallyTested','P.LongTermSideEffects','P.TrustReportsMOH','P.ChipBelief',
    'P.MandatoryBelief','P.HalalDoubt','P.AlternativeMedicineBelief','P.HighRiskJob',
    'P.WorkDemand','A.BoosterProtectSevere','A.BoosterProtectFamily',
    'A.BoosterPreventSpread','A.ReturnDailyActivities','A.RecommendingBooster',
    'A.MaintainAntibody','A.NoSeriousSideEffects','B.RiskWithoutBooster',
    'B.SevereCompWithoutBooster','B.SpreadFamilyWithoutBooster',
    'B.TransmitPatientsWithoutBooster','B.SkepticalBooster','B.BoosterSafe',
    'B.BoosterMoreSideEffects','B.PreferNaturalImmunity','B.EliminateStigma',
    'B.PublicRiskWithoutBooster','C.BoosterNotNeeded','C.BoosterHarm',
    'C.EvidenceInsufficient','C.TrustSocialMedia','C.SideEffectsNoImpact'
]

chi2_columns = [
    'Gender','Education','Income','Pregnant','PatientContact','Occupation',
    'CovidPatientCare','PreExistingCondition','CovidInfected','Severity',
    'AdditionalVaccines','VaccinationStage','VaccinationSideEffects',
    'FamilySideEffects','SideEffectsAffectView',

    # one-hot race dummies
    'Race_Chinese','Race_Indian','Race_Malay','Race_Siam',

    # one-hot religion dummies
    'Religion_Buddha','Religion_Hindu','Religion_Islam',

    # one-hot location dummies
    'Location_Johor','Location_Kedah','Location_Kelantan','Location_Kuantan ',
    'Location_Melaka','Location_Negeri Sembilan','Location_Pahang',
    'Location_Perlis','Location_Pulau Pinang','Location_Sabah',
    'Location_Selangor','Location_Terengganu'
]


# Extract and save ANOVA columns from the training set
anova_train = x_train_normalized[anova_columns]

# Extract and save Chi-Squared columns from the training set
chi2_train = x_train_normalized[chi2_columns]

In [7]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif, chi2

# ----------------------------
# Helper: drop constant columns
# ----------------------------
def drop_constant_columns(X):
    # Keep columns that have more than 1 unique value
    return X.loc[:, X.nunique(dropna=False) > 1]

# ----------------------------
# 1) ANOVA on numeric features
# ----------------------------
anova_train_clean = drop_constant_columns(anova_train)

anova_selector = SelectKBest(score_func=f_classif, k='all')
anova_selector.fit(anova_train_clean, y_train_normalized)

anova_scores = anova_selector.scores_
anova_pvals  = anova_selector.pvalues_

feature_scores_anova = pd.DataFrame({
    "Feature": anova_train_clean.columns,
    "Score": anova_scores,          # F-statistic
    "P-value": anova_pvals,
    "Method": "ANOVA"
})

# ----------------------------
# 2) Chi-square on categorical/non-negative features
# ----------------------------
chi2_train_clean = drop_constant_columns(chi2_train)

# IMPORTANT: chi2 requires non-negative values
if (chi2_train_clean.values < 0).any():
    raise ValueError("Chi-square requires non-negative features. Check your preprocessing for chi2 columns.")

chi_selector = SelectKBest(score_func=chi2, k='all')
chi_selector.fit(chi2_train_clean, y_train_normalized)

chi_scores = chi_selector.scores_     # chi-square statistic
chi_pvals  = chi_selector.pvalues_

feature_scores_chi = pd.DataFrame({
    "Feature": chi2_train_clean.columns,
    "Score": chi_scores,
    "P-value": chi_pvals,
    "Method": "Chi2"
})

# ----------------------------
# 3) Combine + sort baseline ranking
# ----------------------------
feature_scores = pd.concat([feature_scores_anova, feature_scores_chi], ignore_index=True)

# Drop any NaNs just in case
feature_scores = feature_scores.dropna(subset=["P-value"])

# Sort: smallest p-value = most important
feature_scores_sorted = feature_scores.sort_values(by="P-value", ascending=True).reset_index(drop=True)

print(feature_scores_sorted.head(30))

# Save baseline ranking
feature_scores_sorted.to_csv("p_values_for_base.csv", index=False)

                             Feature      Score       P-value Method
0             A.BoosterPreventSpread  27.963764  4.084879e-07  ANOVA
1           S.VaccineImportantHealth  25.686817  1.117009e-06  ANOVA
2             A.BoosterProtectFamily  24.896360  1.589586e-06  ANOVA
3         B.PublicRiskWithoutBooster  24.367642  2.014824e-06  ANOVA
4             A.BoosterProtectSevere  21.565663  7.182520e-06  ANOVA
5           S.VaccineEffectiveSevere  20.630428  1.104197e-05  ANOVA
6              A.RecommendingBooster  19.843313  1.589492e-05  ANOVA
7                S.PreventNewVariant  19.505925  1.859376e-05  ANOVA
8               B.RiskWithoutBooster  19.192424  2.151861e-05  ANOVA
9         B.SevereCompWithoutBooster  18.924258  2.438991e-05  ANOVA
10           B.PreferNaturalImmunity  18.794721  2.591359e-05  ANOVA
11  B.TransmitPatientsWithoutBooster  18.010140  3.745505e-05  ANOVA
12                C.BoosterNotNeeded  17.751025  4.232239e-05  ANOVA
13            S.VaccineBeneficialM

In [8]:
feature_scores_sorted.to_csv('../p_values_for_base.csv')